# Freddie Mac Multi-Year Origination Validation

## Purpose

This notebook applies one reusable Freddie Mac origination pipeline to the
2015–2019 annual sample files. It validates the current 31-field Release 47
layout, preserves the raw fields, creates documented clean and harmonized
fields, profiles annual data quality, compares annual distributions, and
verifies every saved output.

### Why use 2015–2019?

The five consecutive pre-pandemic origination cohorts provide a coherent
development window with two useful properties:

- 2015–2016 provide a baseline before Property Valuation Method disclosure
  begins for 2017 originations.
- The window crosses the 2018 Q2 LTV, CLTV, and Number of Borrowers definition
  changes and includes 2019 as the first full calendar year governed by the
  expanded definitions.
- It captures the decline and completion of HARP through December 2018 while
  keeping the initial reliability study separate from the exceptional
  pandemic-era mortgage environment beginning in 2020.

The files are fixed-size annual samples, so percentages describe these samples
and must not be presented as full-population Freddie Mac estimates.

### Documentation basis

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf)
- [Release 47 disclosure changes](https://www.freddiemac.com/fmac-resources/research/pdf/disclosure-changes-summary.pdf)
- [Historical disclosure changes](https://capitalmarkets.freddiemac.com/crt/docs/pdfs/historical-disclosure-changes.pdf)
- [Single-Family Loan-Level Dataset](https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset)
- [FHFA confirmation of HARP completion on December 31, 2018](https://www.fhfa.gov/news/news-release/fannie-mae-and-freddie-mac-refinance-volume-decreases-in-fourth-quarter-2018)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_cleaning import clean_origination
from src.freddie_ingestion import load_origination
from src.freddie_pipeline import run_origination_pipeline
from src.freddie_quality import create_origination_quality_report
from src.freddie_summary import create_annual_summary
from src.freddie_validation import validate_origination


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

## 1. File Configuration

Define the complete study window once. Every later comparison, validation,
test, save, and read-back loop uses `YEARS`; no analysis cell is tied to a
single annual sample.

In [ ]:
YEARS = [2015, 2016, 2017, 2018, 2019]

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)

ORIGINATION_FILES = {
    year: (
        RAW_DATA_DIRECTORY
        / f"sample_{year}"
        / f"sample_orig_{year}.txt"
    )
    for year in YEARS
}

file_check = pd.DataFrame(
    [
        {
            "year": year,
            "exists": file_path.exists(),
            "path": str(file_path),
        }
        for year, file_path in ORIGINATION_FILES.items()
    ]
)

display(file_check)

missing_files = file_check.loc[
    ~file_check["exists"],
    "path",
].tolist()

if missing_files:
    raise FileNotFoundError(
        "Missing annual origination files: "
        f"{missing_files}"
    )

## 2. Load and Structurally Validate Annual Data

Each pipe-delimited file is scanned before pandas ingestion. The scan verifies
that every record has 31 fields and that no raw value exceeds its documented
Release 47 maximum field length. The loaded shape is then reported for every
year.

In [ ]:
raw_origination = {}
structure_rows = []

for year in YEARS:
    annual_data = load_origination(
        ORIGINATION_FILES[year]
    )

    raw_origination[year] = annual_data

    structure_rows.append(
        {
            "year": year,
            **annual_data.attrs[
                "structure_validation"
            ],
            "loaded_rows": annual_data.shape[0],
            "loaded_columns": annual_data.shape[1],
        }
    )

structure_validation = pd.DataFrame(
    structure_rows
)

display(structure_validation)

## 3. Annual Data-Quality Reports

The quality report separates five different concepts:

1. undocumented categorical codes;
2. documented sentinel frequency;
3. raw null frequency;
4. non-sentinel values outside documented numeric ranges; and
5. time-aware definition and LTV/CLTV rule warnings.

This separation prevents a documented unavailable code from being confused
with an ingestion null or an invalid value.

In [ ]:
quality_reports = {}
quality_check_rows = []

for year in YEARS:
    annual_report = create_origination_quality_report(
        data=raw_origination[year],
        year=year,
    )

    quality_reports[year] = annual_report

    quality_check_rows.append(
        {
            "year": year,
            "unexpected_code_rows": len(
                annual_report["unexpected_codes"]
            ),
            "numeric_range_warning_records": int(
                annual_report[
                    "numeric_range_warnings"
                ]["count"].sum()
            ),
            "definition_warning_records": int(
                annual_report[
                    "definition_warnings"
                ]["count"].sum()
            ),
            "ratio_warning_records": len(
                annual_report["ratio_warnings"]
            ),
        }
    )

quality_check_summary = pd.DataFrame(
    quality_check_rows
)

display(quality_check_summary)

### 3.1 Documented Sentinel Frequencies

Sentinels are documented unavailable, unknown, or not-applicable codes. They
are monitored in raw data and converted to missing only in the corresponding
clean fields. In particular, VantageScore 4.0 is a Release 47 field; historical
sample rows may legitimately contain `9999` throughout.

In [ ]:
sentinel_counts = pd.concat(
    [
        quality_reports[year]["sentinel_counts"]
        for year in YEARS
    ],
    ignore_index=True,
)

nonzero_sentinels = (
    sentinel_counts.loc[
        sentinel_counts["count"] > 0,
        [
            "year",
            "column",
            "sentinel",
            "count",
            "percentage",
        ],
    ]
    .sort_values(
        by=["column", "year"]
    )
    .reset_index(drop=True)
)

display(nonzero_sentinels.round(3))

### 3.2 Property Valuation Method Availability

Usable Property Valuation Method disclosure begins with 2017 originations.
Codes `1`–`3` are usable methods in this study window; code `7` means
unavailable. Code `4` (ACE plus Property Data Report) does not become
effective until 2022 and therefore should not appear in 2015–2019.

In [ ]:
property_valuation_rows = []

for year in YEARS:
    valuation_method = raw_origination[
        year
    ]["PROPERTY VALUATION METHOD"]

    for value, count in (
        valuation_method
        .value_counts(dropna=False)
        .sort_index()
        .items()
    ):
        property_valuation_rows.append(
            {
                "year": year,
                "property_valuation_method": value,
                "count": int(count),
                "percentage": (
                    count
                    / len(valuation_method)
                    * 100
                ),
            }
        )

property_valuation_comparison = pd.DataFrame(
    property_valuation_rows
)

display(
    property_valuation_comparison.round(3)
)

### 3.3 Raw Missingness Comparison

Raw null percentages are aligned across all five years. Documented sentinels
are intentionally excluded from this table because they are reported
separately above. Consecutive-year changes are percentage-point differences.

In [ ]:
missingness = pd.concat(
    [
        quality_reports[year]["missingness"]
        for year in YEARS
    ],
    ignore_index=True,
)

missingness_comparison = (
    missingness.pivot(
        index="column",
        columns="year",
        values="missing_percentage",
    )
    .reset_index()
)

missingness_comparison.columns.name = None

missingness_comparison = (
    missingness_comparison.rename(
        columns={
            year: f"missing_{year}_pct"
            for year in YEARS
        }
    )
)

for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    missingness_comparison[
        (
            f"change_{current_year}"
            f"_minus_{previous_year}_pp"
        )
    ] = (
        missingness_comparison[
            f"missing_{current_year}_pct"
        ]
        - missingness_comparison[
            f"missing_{previous_year}_pct"
        ]
    )

annual_missingness_columns = [
    f"missing_{year}_pct"
    for year in YEARS
]

nonzero_missingness = (
    missingness_comparison.loc[
        missingness_comparison[
            annual_missingness_columns
        ]
        .gt(0)
        .any(axis=1)
    ]
    .sort_values("column")
    .reset_index(drop=True)
)

display(nonzero_missingness.round(3))

### 3.4 Documentation-Aware Warning Details

Numeric-range warnings cover FICO, VantageScore 4.0, mortgage insurance,
Number of Units, DTI, and Number of Borrowers after excluding their documented
sentinels. Definition warnings enforce effective dates for Number of Borrowers
and Property Valuation Method, Pre-HARP linkage consistency, and the documented
DTI-unavailable rule for HARP/Relief Refinance loans. Ratio warnings apply the
quarter-sensitive rules documented below.

In [ ]:
for report_name in [
    "unexpected_codes",
    "numeric_range_warnings",
    "definition_warnings",
    "ratio_warnings",
]:
    annual_tables = [
        quality_reports[year][report_name]
        for year in YEARS
        if not quality_reports[year][
            report_name
        ].empty
    ]

    print(report_name)

    if annual_tables:
        display(
            pd.concat(
                annual_tables,
                ignore_index=True,
            )
        )
    else:
        print("No warnings.")
        print()

## 4. Cleaning and Critical Validation

Cleaning is additive. It preserves all 31 raw fields and appends:

- nine numeric clean fields;
- five categorical clean fields;
- a three-character postal-code clean field; and
- a harmonized Number of Borrowers group.

This produces 47 columns. The harmonized borrower field uses `1` and `2+`
because the raw definition changes in 2018 Q2: earlier values distinguish one
borrower from multiple borrowers, while later values report exact counts from
1 to 10. Raw postal code `000` is documented as unavailable and is therefore
a valid missing value after cleaning, not an invalid postal representation.

Validation checks row preservation, raw-field preservation, required clean
fields, sentinel removal, loan identifiers, postal representation, and the
borrower-group derivation.

In [ ]:
cleaned_origination = {}
validation_rows = []

for year in YEARS:
    cleaned_data = clean_origination(
        raw_origination[year]
    )

    validation_report = validate_origination(
        raw=raw_origination[year],
        cleaned=cleaned_data,
        expected_rows=50000,
    )

    cleaned_origination[year] = cleaned_data

    validation_rows.append(
        {
            "year": year,
            **validation_report,
        }
    )

validation_summary = pd.DataFrame(
    validation_rows
)

display(validation_summary)

## 5. Standardized Annual Summaries

Numeric summaries use cleaned FICO, DTI, LTV, CLTV, mortgage-insurance, and
VantageScore fields so sentinels cannot distort descriptive statistics.
Categorical summaries use cleaned categories and the harmonized borrower
group. These tables are descriptive screening tools; later drift analysis
should apply formal stability tests.

In [ ]:
annual_summaries = {}

for year in YEARS:
    annual_summaries[year] = (
        create_annual_summary(
            data=cleaned_origination[year],
            year=year,
        )
    )

numeric_summary = pd.concat(
    [
        annual_summaries[year]["numeric"]
        for year in YEARS
    ],
    ignore_index=True,
)

categorical_summary = pd.concat(
    [
        annual_summaries[year]["categorical"]
        for year in YEARS
    ],
    ignore_index=True,
)

display(numeric_summary.round(3))

### 5.1 Fixed-Threshold Ratio Comparison

The `>105%` table is retained as a consistent descriptive trend measure
across years. It is **not** a universal validity rule. Freddie Mac's documented
origination rules are:

| Period / population | LTV range | CLTV range |
|---|---:|---:|
| Non-HARP, 2018 Q1 and earlier | 6–105 | 6–200 |
| HARP, 2018 Q1 and earlier | 1–998 | 1–998 |
| 2018 Q2 and later | 1–998 | 1–998 |

Code `999` means unavailable. Freddie Mac also directs that CLTV be reported
as `999` if calculated CLTV is lower than LTV. The separate ratio-warning
report applies these rules at record level.

In [ ]:
high_ratio_summary_rows = []
extreme_ratio_records_by_year = {}

for year in YEARS:
    data = cleaned_origination[year]

    ltv = data[
        "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
    ]

    cltv = data[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
    ]

    high_ratio_mask = (
        ltv.gt(105)
        | cltv.gt(105)
    )

    high_ratio_records = data.loc[
        high_ratio_mask
    ].copy()

    extreme_ratio_records_by_year[
        year
    ] = high_ratio_records

    pre_harp_sequence = (
        high_ratio_records[
            "PRE-HARP LOAN SEQUENCE NUMBER"
        ]
        .astype("string")
        .str.strip()
    )

    pre_harp_present = (
        pre_harp_sequence.notna()
        & pre_harp_sequence.ne("")
    )

    high_ratio_summary_rows.append(
        {
            "year": year,
            "records_above_105": len(
                high_ratio_records
            ),
            "percentage_above_105": (
                len(high_ratio_records)
                / len(data)
                * 100
            ),
            "ltv_above_105": int(
                ltv.gt(105).sum()
            ),
            "cltv_above_105": int(
                cltv.gt(105).sum()
            ),
            "harp_y": int(
                high_ratio_records[
                    "HARP INDICATOR"
                ].eq("Y").sum()
            ),
            "pre_harp_id_present": int(
                pre_harp_present.sum()
            ),
            "maximum_ltv": ltv.max(),
            "maximum_cltv": cltv.max(),
        }
    )

high_ratio_summary = pd.DataFrame(
    high_ratio_summary_rows
)

display(high_ratio_summary.round(3))

### 5.2 Record-Level Review of Extreme Ratios

Display the ten highest LTV/CLTV records for every year. This retains actual
loan identifiers for traceability and avoids replacing real observations with
test identifiers. The synthetic identifiers used later are confined to the
unit-style rule test.

In [ ]:
extreme_record_columns = [
    "LOAN IDENTIFIER",
    "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
    "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
    "LOAN PURPOSE",
    "HARP INDICATOR",
    "PRE-HARP LOAN SEQUENCE NUMBER",
    "SPECIAL ELIGIBILITY PROGRAM",
    "PROPERTY VALUATION METHOD",
    "PROPERTY VALUATION METHOD_CLEAN",
    "ORIGINAL UPB",
    "PROPERTY TYPE",
    "OCCUPANCY STATUS",
]

for year in YEARS:
    extreme_records = (
        extreme_ratio_records_by_year[year]
        .loc[:, extreme_record_columns]
        .sort_values(
            by=[
                "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
                "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
            ],
            ascending=False,
        )
        .head(10)
        .reset_index(drop=True)
    )

    print(f"Year: {year}")
    display(extreme_records)

### 5.3 Actual Ratio-Rule Warnings

These are documentation-aware exceptions, not simply all loans above 105%.
An empty table means the observed ratios comply with the applicable
year-quarter, HARP, sentinel, and CLTV-versus-LTV rules.

In [ ]:
ratio_warning_tables = [
    quality_reports[year]["ratio_warnings"]
    for year in YEARS
]

ratio_warnings = pd.concat(
    ratio_warning_tables,
    ignore_index=True,
)

print(
    "Total ratio-rule warning records:",
    len(ratio_warnings),
)

display(ratio_warnings)

### 5.4 Synthetic Ratio-Rule Test

A synthetic record is a temporary copy whose fields are deliberately changed
to test a rule. It is never written into the source sample or cleaned output.
Each test case has its own identifier and expected result, which is more
reliable than checking only a total warning count.

In [ ]:
quality_rule_test_rows = []
quality_rule_warning_tables = []

for year in YEARS:
    year_suffix = str(year)[-2:]

    test_cases = [
        {
            "case": "q1_non_harp_120",
            "quarter": 1,
            "ltv": 120,
            "cltv": 120,
            "expected_warning": year <= 2018,
        },
        {
            "case": "q2_non_harp_120",
            "quarter": 2,
            "ltv": 120,
            "cltv": 120,
            "expected_warning": year < 2018,
        },
        {
            "case": "above_absolute_maximum",
            "quarter": 2,
            "ltv": 1000,
            "cltv": 1000,
            "expected_warning": True,
        },
        {
            "case": "below_minimum",
            "quarter": 2,
            "ltv": 0,
            "cltv": 0,
            "expected_warning": True,
        },
        {
            "case": "cltv_below_ltv",
            "quarter": 2,
            "ltv": 80,
            "cltv": 70,
            "expected_warning": True,
        },
    ]

    quality_rule_test = (
        raw_origination[year]
        .iloc[:len(test_cases)]
        .copy()
    )

    test_identifiers = []

    for position, test_case in enumerate(
        test_cases,
        start=1,
    ):
        test_identifiers.append(
            (
                f"T{year_suffix}Q"
                f"{test_case['quarter']}"
                f"{position:07d}"
            )
        )

    quality_rule_test[
        "LOAN IDENTIFIER"
    ] = test_identifiers

    quality_rule_test[
        "ORIGINAL LOAN-TO-VALUE (LTV)"
    ] = [
        test_case["ltv"]
        for test_case in test_cases
    ]

    quality_rule_test[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)"
    ] = [
        test_case["cltv"]
        for test_case in test_cases
    ]

    quality_rule_test["HARP INDICATOR"] = "N"

    quality_rule_test[
        "PRE-HARP LOAN SEQUENCE NUMBER"
    ] = pd.NA

    test_report = (
        create_origination_quality_report(
            data=quality_rule_test,
            year=year,
        )
    )

    detected_warnings = test_report[
        "ratio_warnings"
    ]

    detected_identifiers = set(
        detected_warnings[
            "LOAN IDENTIFIER"
        ].astype("string")
    )

    for test_case, identifier in zip(
        test_cases,
        test_identifiers,
    ):
        detected_warning = (
            identifier in detected_identifiers
        )

        quality_rule_test_rows.append(
            {
                "year": year,
                "case": test_case["case"],
                "expected_warning": (
                    test_case[
                        "expected_warning"
                    ]
                ),
                "detected_warning": (
                    detected_warning
                ),
                "test_passed": (
                    detected_warning
                    == test_case[
                        "expected_warning"
                    ]
                ),
            }
        )

    quality_rule_warning_tables.append(
        detected_warnings
    )

quality_rule_test_results = pd.DataFrame(
    quality_rule_test_rows
)

quality_rule_test_summary = (
    quality_rule_test_results
    .groupby("year", as_index=False)
    .agg(
        expected_warnings=(
            "expected_warning",
            "sum",
        ),
        detected_warnings=(
            "detected_warning",
            "sum",
        ),
        test_passed=(
            "test_passed",
            "all",
        ),
    )
)

quality_rule_test_warnings = pd.concat(
    quality_rule_warning_tables,
    ignore_index=True,
)

display(quality_rule_test_summary)
display(quality_rule_test_warnings)

failed_quality_rule_tests = (
    quality_rule_test_results.loc[
        ~quality_rule_test_results[
            "test_passed"
        ]
    ]
)

assert failed_quality_rule_tests.empty, (
    "Ratio-rule tests failed:\n"
    f"{failed_quality_rule_tests}"
)

## 6. Categorical Distribution Comparison

Category percentages are aligned across 2015–2019. Each annual percentage has
an explicit year in its column name, followed by consecutive-year
percentage-point changes. Number of Borrowers uses the comparable `1` versus
`2+` grouping rather than mixing the pre- and post-2018 Q2 raw definitions.

In [ ]:
categorical_comparison = (
    categorical_summary.pivot_table(
        index=[
            "variable",
            "category",
        ],
        columns="year",
        values="percentage",
        fill_value=0,
    )
    .reset_index()
)

categorical_comparison.columns.name = None

categorical_comparison = (
    categorical_comparison.rename(
        columns={
            year: f"percentage_{year}"
            for year in YEARS
        }
    )
)

categorical_change_columns = []

for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    change_column = (
        f"change_{current_year}"
        f"_minus_{previous_year}_pp"
    )

    categorical_comparison[
        change_column
    ] = (
        categorical_comparison[
            f"percentage_{current_year}"
        ]
        - categorical_comparison[
            f"percentage_{previous_year}"
        ]
    )

    categorical_change_columns.append(
        change_column
    )

categorical_comparison[
    "maximum_absolute_change_pp"
] = (
    categorical_comparison[
        categorical_change_columns
    ]
    .abs()
    .max(axis=1)
)

largest_categorical_changes = (
    categorical_comparison
    .sort_values(
        "maximum_absolute_change_pp",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)

display(
    largest_categorical_changes.round(3)
)

## 7. Save Verified 2015–2019 Outputs

Run the end-to-end pipeline for every configured year. Each annual run saves
the cleaned Parquet data, numeric and categorical summaries, sentinel and
missingness reports, categorical-code warnings, numeric-range warnings,
time-aware definition warnings, ratio warnings, and a validation JSON file.

In [ ]:
PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

pipeline_results = {}

for year in YEARS:
    pipeline_result = run_origination_pipeline(
        year=year,
        input_file=ORIGINATION_FILES[year],
        output_directory=(
            PROCESSED_DATA_DIRECTORY
        ),
        expected_rows=50000,
    )

    pipeline_results[year] = pipeline_result

    print(f"Year: {year}")
    print(
        "Structure validation:",
        pipeline_result[
            "structure_validation"
        ],
    )
    print(
        "Data validation:",
        pipeline_result["validation"],
    )
    print("Saved outputs:")

    for output_name, output_path in (
        pipeline_result[
            "output_paths"
        ].items()
    ):
        print(
            f"{output_name}: {output_path}"
        )

    print("=" * 70)

### 7.1 Saved Report Verification

Read every saved quality CSV and its validation JSON. The row counts must agree
for each report type and year.

In [ ]:
report_json_keys = {
    "unexpected_codes": (
        "unexpected_code_rows"
    ),
    "numeric_range_warnings": (
        "numeric_range_warning_rows"
    ),
    "definition_warnings": (
        "definition_warning_rows"
    ),
    "ratio_warnings": (
        "ratio_warning_rows"
    ),
}

saved_output_check_rows = []

for year in YEARS:
    validation_path = pipeline_results[
        year
    ]["output_paths"]["validation"]

    with validation_path.open(
        mode="r",
        encoding="utf-8",
    ) as validation_file:
        saved_validation = json.load(
            validation_file
        )

    for report_name, json_key in (
        report_json_keys.items()
    ):
        saved_report = pd.read_csv(
            pipeline_results[
                year
            ]["output_paths"][report_name]
        )

        saved_output_check_rows.append(
            {
                "year": year,
                "report": report_name,
                "saved_csv_rows": len(
                    saved_report
                ),
                "validation_json_rows": (
                    saved_validation[
                        json_key
                    ]
                ),
                "row_counts_match": (
                    len(saved_report)
                    == saved_validation[
                        json_key
                    ]
                ),
            }
        )

saved_output_check = pd.DataFrame(
    saved_output_check_rows
)

display(saved_output_check)

assert saved_output_check[
    "row_counts_match"
].all(), "Saved quality-report row counts do not match."

## 8. Saved-Output Verification

Read each saved Parquet dataset and run the same validation again. This checks
that serialization preserved 50,000 rows, all 31 raw fields, all 16 added
fields, loan identifiers, postal-code representation, and the harmonized
borrower group.

In [ ]:
saved_cleaned_origination = {}
readback_rows = []

for year in YEARS:
    saved_cleaned_data = pd.read_parquet(
        pipeline_results[
            year
        ]["output_paths"]["cleaned"]
    )

    readback_validation = validate_origination(
        raw=raw_origination[year],
        cleaned=saved_cleaned_data,
        expected_rows=50000,
    )

    saved_cleaned_origination[
        year
    ] = saved_cleaned_data

    readback_rows.append(
        {
            "year": year,
            "saved_shape": (
                saved_cleaned_data.shape
            ),
            "postal_code_dtype": str(
                saved_cleaned_data[
                    "POSTAL CODE_CLEAN"
                ].dtype
            ),
            **readback_validation,
        }
    )

readback_summary = pd.DataFrame(
    readback_rows
)

display(readback_summary)

## 9. Conclusion

After a fresh top-to-bottom execution, this section will be replaced with the
final evidence-based 2015–2019 findings. The final conclusion will distinguish:

- ingestion and structural validity;
- documented availability and definition changes;
- actual data-quality exceptions;
- descriptive annual population or product-mix changes; and
- questions reserved for formal drift and model-reliability analysis.

No 2019 result should be inferred from 2015–2018 output; the conclusion must be
written only after all five annual samples and saved-output checks are present.